# Lab 6: Object Detection with YOLO

*Written by Aryan Mikaeili, modified by Yalda.*

Last week a CNN looked at an image and gave **one label**. Today it will find **every object** in the image:
*what* it is and *where* it is (a box). We will build a small version of **YOLO** ("You Only Look Once") from scratch.

1. **What is object detection?** Try a real YOLO model.
2. **The YOLO idea: a grid.** How a box becomes numbers the network can learn.
3. **IoU.** How do we measure if a box is good?
4. **The network.** Your CNN from Lab 5, with a new output.
5. **Loss and training.** Train your own tiny YOLO.
6. **Non-max suppression.** Remove duplicate boxes.
7. **Try it yourself.**

| Part | Time |
|---|---|
| Setup + 1. What is detection? | 15 min |
| 2. The YOLO grid | 25 min |
| 3. IoU | 15 min |
| 4. The network | 15 min |
| 5. Loss and training | 25 min |
| 6. Non-max suppression | 20 min |
| 7. Try it yourself | 5 min |

For each **TODO**, write your code, then run the check cell below it. If you are stuck, open the hidden **Solution** cell.

## Setup

In [ ]:
# install the real YOLO (only needed once, takes ~30 seconds)
%pip install -q ultralytics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torchvision

from ultralytics import YOLO
from ultralytics.utils import ASSETS   # example photos that come with ultralytics

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():   # Apple GPU
    device = "mps"
else:
    device = "cpu"
print("device:", device)

## 1. What is object detection?

| Task | Output |
|---|---|
| **Classification** (Lab 5) | one label for the whole image: *"bus"* |
| **Object detection** (today) | for **every** object: a label, a **box**, and a **confidence** |

Let's run a real, pretrained YOLO model (YOLO11, trained on the COCO dataset with 80 classes).

In [ ]:
yolo = YOLO("yolo11n.pt")          # "n" = nano, the smallest version (downloads ~5 MB)
photo = Image.open(ASSETS / "bus.jpg")

result = yolo(photo, verbose=False)[0]
print("boxes (x1, y1, x2, y2) in pixels:\n", result.boxes.xyxy.round())
print("confidence:", result.boxes.conf.round(decimals=2))
print("class:     ", [result.names[int(c)] for c in result.boxes.cls])

plt.figure(figsize=(6, 8))
plt.imshow(result.plot()[:, :, ::-1])   # result.plot() draws the boxes for us (in BGR colors, so we flip to RGB)
plt.axis("off")
plt.show()

One look at the image (one pass through the network) and we get all the boxes. That is the "You Only Look Once" idea.
Older methods (sliding windows, R-CNN) ran a classifier on **thousands** of crops of the image, which is very slow.

**How does it work?** By the end of this lab you will have built and trained your own tiny version.

**YOLO running live on video (fun, no need to watch it all)** (click to watch)

<a href="https://www.youtube.com/watch?v=VOC3huqHrss" target="_blank"><img src="https://img.youtube.com/vi/VOC3huqHrss/hqdefault.jpg" width="480"></a>

### Two ways to write a box

All coordinates are **normalized**: divided by the image size, so they go from 0 to 1.

- **Corners** `[x1, y1, x2, y2]`: top-left and bottom-right corner. Good for drawing and for IoU.
- **Center** `[x, y, w, h]`: center point, width and height. This is what YOLO predicts.

Remember: in images, $y$ goes **down**. $(0, 0)$ is the top-left corner.

**TODO 1:** convert a box from center format to corner format.

In [ ]:
def xywh_to_xyxy(box):
    x, y, w, h = box
    # TODO: compute the corners (hint: the left edge is half a width to the left of the center)
    x1 = ...
    y1 = ...
    x2 = ...
    y2 = ...
    return [x1, y1, x2, y2]

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def xywh_to_xyxy(box):
    x, y, w, h = box
    x1 = x - w / 2
    y1 = y - h / 2
    x2 = x + w / 2
    y2 = y + h / 2
    return [x1, y1, x2, y2]

In [ ]:
# run this cell to check your code
mine = xywh_to_xyxy([0.5, 0.4, 0.2, 0.6])
print(mine)
print("correct:", np.allclose(mine, [0.4, 0.1, 0.6, 0.7]))

## 2. The YOLO idea: a grid

### Our dataset: shapes

Real photos need a big network and hours of training. To see every step clearly, we make our own small dataset:
**112×112** images with 1 to 4 shapes. **3 classes:** circle, square, triangle.
The code below draws the images and saves the box of every shape (center format, normalized). You can skip reading it.

In [ ]:
IMG = 112                                  # image size in pixels
CLASSES = ["circle", "square", "triangle"]
C = len(CLASSES)
S = 7                                      # the grid is S x S (like the YOLO paper)


def make_image(rng):
    background = tuple(int(v) for v in rng.integers(170, 256, 3))
    img = Image.new("RGB", (IMG, IMG), background)
    draw = ImageDraw.Draw(img)
    boxes, labels, used_cells = [], [], []
    for _ in range(rng.integers(1, 5)):
        for attempt in range(20):
            size = rng.integers(16, 48)
            cx = rng.uniform(size / 2, IMG - size / 2)
            cy = rng.uniform(size / 2, IMG - size / 2)
            cell = (int(cy / IMG * S), int(cx / IMG * S))
            new = xywh_to_xyxy([cx, cy, size, size])
            far_enough = all(abs(cx - b[0] * IMG) > (size + b[2] * IMG) / 2 or abs(cy - b[1] * IMG) > (size + b[3] * IMG) / 2
                             for b in boxes)
            if cell not in used_cells and far_enough:   # at most one object center per grid cell, no overlaps
                break
        else:
            continue
        c = rng.integers(0, C)
        color = tuple(int(v) for v in rng.integers(0, 140, 3))
        if CLASSES[c] == "circle":
            draw.ellipse(new, fill=color)
        elif CLASSES[c] == "square":
            draw.rectangle(new, fill=color)
        else:
            draw.polygon([(cx, new[1]), (new[0], new[3]), (new[2], new[3])], fill=color)
        boxes.append([cx / IMG, cy / IMG, size / IMG, size / IMG])
        labels.append(int(c))
        used_cells.append(cell)
    return np.array(img), boxes, labels


def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    images, all_boxes, all_labels = [], [], []
    for _ in range(n):
        img, boxes, labels = make_image(rng)
        images.append(img)
        all_boxes.append(boxes)
        all_labels.append(labels)
    return np.array(images), all_boxes, all_labels


train_images, train_boxes, train_labels = make_dataset(4000, seed=0)
test_images, test_boxes, test_labels = make_dataset(300, seed=1)
print("train:", train_images.shape, " test:", test_images.shape)
print("boxes of image 0 (x, y, w, h):", np.round(train_boxes[0], 2))
print("labels of image 0:", [CLASSES[c] for c in train_labels[0]])

In [ ]:
COLORS = ["tab:red", "tab:blue", "tab:green"]


def draw_boxes(ax, boxes, labels, scores=None):
    # boxes: center format, normalized. Draws them on an image of size IMG.
    for i in range(len(boxes)):
        x1, y1, x2, y2 = [v * IMG for v in xywh_to_xyxy(boxes[i])]
        color = COLORS[labels[i]]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2))
        text = CLASSES[labels[i]]
        if scores is not None:
            text += f" {scores[i]:.2f}"
        ax.text(x1, y1 - 2, text, color=color, fontsize=8, fontweight="bold")


fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_images[i])
    draw_boxes(ax, train_boxes[i], train_labels[i])
    ax.axis("off")
plt.tight_layout()
plt.show()

### The grid

YOLO cuts the image into an $S \times S$ grid (here $7 \times 7$, each cell is 16×16 pixels).

> **Rule:** the cell that contains the **center** of an object is **responsible** for that object.

Every cell predicts the same list of numbers:

| | meaning |
|---|---|
| `conf` | is there an object center in me? (1 = yes, 0 = no) |
| `x, y` | where is the center, **inside my cell** (0 to 1) |
| `w, h` | width and height of the box, as a fraction of the **whole image** |
| `circle, square, triangle` | which class (one-hot: `[0, 1, 0]` = square) |

So the network output is $S \times S \times (5 + C) = 7 \times 7 \times 8$ numbers. Detection becomes
"classification + a few numbers" for every cell, which a CNN can learn!

*(The paper uses 2 boxes per cell and 20 classes: $7 \times 7 \times 30$. We use 1 box per cell to keep it simple.)*

In [ ]:
i = 3   # image number (Python counts from 0, so this is the 4th image)
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(train_images[i])
for k in range(1, S):                                    # grid lines
    ax.axhline(k * IMG / S, color="black", linewidth=0.5)
    ax.axvline(k * IMG / S, color="black", linewidth=0.5)
cell = IMG / S   # size of one grid cell in pixels (112 / 7 = 16)
# train_boxes[i] = list of ALL boxes in image i, train_labels[i] = their classes
for (x, y, w, h), c in zip(train_boxes[i], train_labels[i]):   # one object at a time
    col = int(x * S)
    row = int(y * S)
    ax.add_patch(patches.Rectangle((col * cell, row * cell), cell, cell, color=COLORS[c], alpha=0.4))  # responsible cell
    ax.plot(x * IMG, y * IMG, "o", color="yellow", markeredgecolor="black")                            # center
draw_boxes(ax, train_boxes[i], train_labels[i])
ax.set_title("yellow dot = center, filled cell = responsible cell")
ax.axis("off")
plt.show()

**Watch (7 min):** Andrew Ng explains the same idea. He uses a 3×3 grid; we use 7×7.

**Andrew Ng: YOLO Algorithm** (click to watch)

<a href="https://www.youtube.com/watch?v=9s_FpMpdYW8" target="_blank"><img src="https://img.youtube.com/vi/9s_FpMpdYW8/hqdefault.jpg" width="480"></a>

### From boxes to the target grid

To train, we need the **target**: the $7 \times 7 \times 8$ numbers we want the network to output for an image.

For a box with center $(x, y)$ (normalized), the responsible cell is

$$\text{col} = \lfloor x \cdot S \rfloor, \qquad \text{row} = \lfloor y \cdot S \rfloor$$

and the center **inside** that cell is

$$x_\text{cell} = x \cdot S - \text{col}, \qquad y_\text{cell} = y \cdot S - \text{row}$$

Example: $x = 0.5$, $S = 7$: $x \cdot S = 3.5$, so col $= 3$ and $x_\text{cell} = 0.5$ (the middle of cell 3).

**TODO 2:** fill in `encode_target`. Every cell starts as all zeros. For each box, fill in its responsible cell.
The order inside a cell is `[conf, x_cell, y_cell, w, h, circle, square, triangle]`.

In [ ]:
def encode_target(boxes, labels):
    target = torch.zeros(S, S, 5 + C)
    for (x, y, w, h), c in zip(boxes, labels):
        # TODO: find the responsible cell (hint: int() rounds down)
        col = ...
        row = ...
        # TODO: the center inside the cell
        x_cell = ...
        y_cell = ...
        # TODO: fill in the cell. Note: target[row, col] (row first, like a matrix)
        target[row, col, 0] = ...                 # conf
        target[row, col, 1:5] = torch.tensor(...) # x_cell, y_cell, w, h
        target[row, col, 5 + c] = ...             # one-hot class
    return target

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def encode_target(boxes, labels):
    target = torch.zeros(S, S, 5 + C)
    for (x, y, w, h), c in zip(boxes, labels):
        col = int(x * S)
        row = int(y * S)
        x_cell = x * S - col
        y_cell = y * S - row
        target[row, col, 0] = 1
        target[row, col, 1:5] = torch.tensor([x_cell, y_cell, w, h])
        target[row, col, 5 + c] = 1
    return target

In [ ]:
# run this cell to check your code
t = encode_target([[0.5, 0.2, 0.3, 0.1]], [2])
print("cell (row 1, col 3):", t[1, 3])
expected = torch.tensor([1, 0.5, 0.4, 0.3, 0.1, 0, 0, 1])
print("correct:", torch.allclose(t[1, 3], expected) and t.sum().item() == expected.sum().item())

### From the grid back to boxes

The network will output a grid like this. To get boxes back, we do the **opposite** of `encode_target`:
for every cell with a high `conf`, $x = (\text{col} + x_\text{cell}) / S$ and $y = (\text{row} + y_\text{cell}) / S$.
**TODO 3:** fill in `decode`. The class is the position of the largest class score (`argmax`).

In [ ]:
def decode(grid, conf_threshold=0.5):
    # grid: (S, S, 5 + C). Returns boxes (center format), labels and scores of the cells with conf > threshold
    boxes, labels, scores = [], [], []
    for row in range(S):
        for col in range(S):
            conf = grid[row, col, 0].item()
            if conf > conf_threshold:
                x_cell, y_cell, w, h = grid[row, col, 1:5].tolist()
                # TODO: the center in the whole image (the opposite of encode_target)
                x = ...
                y = ...
                # TODO: the class = index of the largest of the class scores grid[row, col, 5:] (hint: .argmax().item())
                label = ...
                boxes.append([x, y, w, h])
                labels.append(label)
                scores.append(conf)
    return boxes, labels, scores

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def decode(grid, conf_threshold=0.5):
    # grid: (S, S, 5 + C). Returns boxes (center format), labels and scores of the cells with conf > threshold
    boxes, labels, scores = [], [], []
    for row in range(S):
        for col in range(S):
            conf = grid[row, col, 0].item()
            if conf > conf_threshold:
                x_cell, y_cell, w, h = grid[row, col, 1:5].tolist()
                x = (col + x_cell) / S
                y = (row + y_cell) / S
                label = grid[row, col, 5:].argmax().item()
                boxes.append([x, y, w, h])
                labels.append(label)
                scores.append(conf)
    return boxes, labels, scores

In [ ]:
# run this cell to check your code
# round trip: boxes -> target grid -> boxes. We should get the same boxes back.
i = 3
boxes, labels, scores = decode(encode_target(train_boxes[i], train_labels[i]))
print("original:", np.round(train_boxes[i], 3).tolist(), train_labels[i])
print("decoded: ", np.round(boxes, 3).tolist(), labels)

ok = True
for i in range(100):
    boxes, labels, scores = decode(encode_target(train_boxes[i], train_labels[i]))
    mine = sorted([b + [l] for b, l in zip(boxes, labels)])
    true = sorted([b + [l] for b, l in zip(train_boxes[i], train_labels[i])])
    ok = ok and len(mine) == len(true) and np.allclose(mine, true, atol=1e-4)
print("correct:", ok)

Now turn the whole dataset into tensors: images as `(N, 3, 112, 112)` with values in [0, 1], and targets as `(N, 7, 7, 8)`
using your `encode_target`.

In [ ]:
def to_tensors(images, all_boxes, all_labels):
    x = torch.tensor(images).permute(0, 3, 1, 2).float() / 255   # (N, 112, 112, 3) -> (N, 3, 112, 112), 0..1
    y = torch.stack([encode_target(b, l) for b, l in zip(all_boxes, all_labels)])
    return x, y


x_train, y_train = to_tensors(train_images, train_boxes, train_labels)
x_test, y_test = to_tensors(test_images, test_boxes, test_labels)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
print("images:", x_train.shape, " targets:", y_train.shape)

## 3. Intersection over Union (IoU)

How good is a predicted box? We compare it with the true box:

$$\text{IoU} = \frac{\text{area of overlap}}{\text{area of union}}$$

IoU = 1: perfect match. IoU = 0: no overlap. A detection usually counts as correct if IoU ≥ 0.5.

In [ ]:
def show_two_boxes(box1, box2, title=""):
    # boxes in corner format, normalized
    fig, ax = plt.subplots(figsize=(3.5, 3.5))
    for box, color in [(box1, "tab:blue"), (box2, "tab:orange")]:
        ax.add_patch(patches.Rectangle(box[:2], box[2] - box[0], box[3] - box[1], color=color, alpha=0.35))
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    if x2 > x1 and y2 > y1:
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, hatch="///", edgecolor="black"))
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)   # y goes down, like in an image
    ax.set_title(title)
    plt.show()


show_two_boxes([0.1, 0.1, 0.6, 0.6], [0.3, 0.3, 0.8, 0.8], "hatched = overlap (intersection)")

**Andrew Ng: Intersection Over Union (4 min, optional)** (click to watch)

<a href="https://www.youtube.com/watch?v=ANIzQ5G-XPE" target="_blank"><img src="https://img.youtube.com/vi/ANIzQ5G-XPE/hqdefault.jpg" width="480"></a>

**TODO 4:** write `iou` for two boxes in **corner** format.

The overlap is also a rectangle. Its left edge is the **larger** of the two left edges, its right edge is the **smaller**
of the two right edges (same for top and bottom). If the boxes do not touch, the width or height comes out negative:
use `max(0, ...)` to make it 0.

In [ ]:
def iou(box1, box2):
    # box1, box2: [x1, y1, x2, y2]
    # TODO: corners of the overlap rectangle (use max and min)
    left = ...
    top = ...
    right = ...
    bottom = ...
    # TODO: area of the overlap (0 if they do not overlap)
    intersection = ...
    # TODO: area of each box, then the union
    area1 = ...
    area2 = ...
    union = ...
    return intersection / union

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def iou(box1, box2):
    # box1, box2: [x1, y1, x2, y2]
    left = max(box1[0], box2[0])
    top = max(box1[1], box2[1])
    right = min(box1[2], box2[2])
    bottom = min(box1[3], box2[3])
    intersection = max(0, right - left) * max(0, bottom - top)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / union

In [ ]:
# run this cell to check your code (compare with torchvision)
rng = np.random.default_rng(0)
ok = True
for _ in range(100):
    a = np.sort(rng.random((2, 2)), axis=0).T.flatten()[[0, 2, 1, 3]].tolist()   # random valid box
    b = np.sort(rng.random((2, 2)), axis=0).T.flatten()[[0, 2, 1, 3]].tolist()
    real = torchvision.ops.box_iou(torch.tensor([a]), torch.tensor([b])).item()
    ok = ok and abs(iou(a, b) - real) < 1e-5
print("IoU of the example above:", round(iou([0.1, 0.1, 0.6, 0.6], [0.3, 0.3, 0.8, 0.8]), 3))
print("correct:", ok)

**Try it:** change the numbers below. How far can you move the orange box before IoU drops below 0.5?

In [ ]:
box1 = [0.2, 0.2, 0.6, 0.6]
box2 = [0.3, 0.3, 0.7, 0.7]
# the math for these two boxes:
#   overlap:      left = max(0.2, 0.3) = 0.3   top = max(0.2, 0.3) = 0.3
#                 right = min(0.6, 0.7) = 0.6  bottom = min(0.6, 0.7) = 0.6
#   intersection: (0.6 - 0.3) * (0.6 - 0.3) = 0.3 * 0.3 = 0.09
#   areas:        box1 = 0.4 * 0.4 = 0.16      box2 = 0.4 * 0.4 = 0.16
#   union:        0.16 + 0.16 - 0.09 = 0.23    (the overlap was counted twice)
#   IoU:          0.09 / 0.23 = 0.39
show_two_boxes(box1, box2, f"IoU = {iou(box1, box2):.2f}")

## 4. The network

It is the CNN you built in Lab 5. Only the end is new: instead of `Linear(..., 10)`, we want a
**7 × 7 grid with 8 numbers per cell**.

Four conv → ReLU → pool blocks shrink the image $112 \to 56 \to 28 \to 14 \to 7$. Now the feature map is already
$7 \times 7$: **one position per grid cell**, each with 128 features. We reorder it to $7 \times 7 \times 128$
(one feature vector per cell), and a **head** `Linear(128, 8)` turns every cell's 128 features into its 8 numbers.
The same Linear layer is used for all 49 cells.

**TODO 5:** build this network (all convs are 3×3 with padding 1):

| Layer | Output shape |
|---|---|
| input | 3 × 112 × 112 |
| block 1: `Conv2d(3, 16)` → ReLU → pool | 16 × 56 × 56 |
| block 2: `Conv2d(16, 32)` → ReLU → pool | 32 × 28 × 28 |
| block 3: `Conv2d(32, 64)` → ReLU → pool | 64 × 14 × 14 |
| block 4: `Conv2d(64, 128)` → ReLU → pool | 128 × 7 × 7 |
| `Conv2d(128, 128)` → ReLU (no pool) | 128 × 7 × 7 |
| reorder to one vector per cell | 7 × 7 × 128 |
| head: `Linear(128, 8)` | 7 × 7 × 8 |

At the end, a **sigmoid** squashes `conf, x, y, w, h` into 0..1 (they are all between 0 and 1). The class scores stay raw,
like in Lab 5.

In [ ]:
class TinyYOLO(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        # TODO: conv2 (16 -> 32), conv3 (32 -> 64), conv4 (64 -> 128), kernel 3, padding 1
        self.conv2 = ...
        self.conv3 = ...
        self.conv4 = ...
        self.conv5 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        # TODO: the head, a Linear layer: 128 features -> 5 + C numbers
        self.head = ...
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        # block 1: 3 x 112 x 112 -> 16 x 56 x 56
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        # TODO: blocks 2, 3 and 4 (-> 128 x 7 x 7), same three steps each



        x = self.conv5(x)
        x = self.relu(x)
        x = x.permute(0, 2, 3, 1)     # (N, 128, 7, 7) -> (N, 7, 7, 128): one feature vector per cell
        x = self.head(x)              # (N, 7, 7, 8): 8 numbers per cell
        boxes = torch.sigmoid(x[..., :5])   # conf, x, y, w, h -> between 0 and 1
        class_scores = x[..., 5:]
        return torch.cat([boxes, class_scores], dim=-1)

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

class TinyYOLO(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.head = nn.Linear(128, 5 + C)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        # block 1: 3 x 112 x 112 -> 16 x 56 x 56
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        # block 2: -> 32 x 28 x 28
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        # block 3: -> 64 x 14 x 14
        x = self.conv3(x)
        x = self.relu(x)
        x = self.pool(x)
        # block 4: -> 128 x 7 x 7
        x = self.conv4(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv5(x)
        x = self.relu(x)
        x = x.permute(0, 2, 3, 1)     # (N, 128, 7, 7) -> (N, 7, 7, 128): one feature vector per cell
        x = self.head(x)              # (N, 7, 7, 8): 8 numbers per cell
        boxes = torch.sigmoid(x[..., :5])   # conf, x, y, w, h -> between 0 and 1
        class_scores = x[..., 5:]
        return torch.cat([boxes, class_scores], dim=-1)

In [ ]:
# run this cell to check your code
m = TinyYOLO()
out = m(torch.zeros(2, 3, IMG, IMG))
n_params = sum(p.numel() for p in m.parameters())
print("output shape:", out.shape, " parameters:", n_params)
print("correct:", out.shape == (2, S, S, 5 + C) and n_params == 246056)

## 5. Loss and training

The YOLO loss adds up a few **squared errors**. For cells **with** an object, we want the right box, `conf` = 1
and the right class. For cells **without** an object, we only want `conf` = 0:

$$
\text{loss} =
\underbrace{5 \sum_{\text{obj}} \big[(x - \hat x)^2 + (y - \hat y)^2 + (\sqrt{w} - \sqrt{\hat w})^2 + (\sqrt{h} - \sqrt{\hat h})^2\big]}_{\text{box}}
+ \underbrace{\sum_{\text{obj}} (\text{conf} - 1)^2}_{\text{"there is an object"}}
+ \underbrace{0.5 \sum_{\text{no obj}} (\text{conf} - 0)^2}_{\text{"there is nothing"}}
+ \underbrace{\text{cross-entropy}_\text{obj}}_{\text{class}}
$$

Why the extra pieces?
- **×5 for the box:** the box is what we care about most, so it gets a bigger weight.
- **×0.5 for empty cells:** most cells are empty (about 46 of 49). Without this, the network would learn to always say "nothing".
- **√w, √h:** an error of 5 pixels is bad for a small box but fine for a big one. The square root makes small boxes count more.
- **Class:** the paper uses a squared error here too. We use cross-entropy, like in Lab 5.

**TODO 6:** write the loss. `p` and `t` hold only the cells **with** an object, `p_empty` and `t_empty` the cells
**without**. `xy_loss` and `class_loss` are done; follow the same pattern for the others.

In [ ]:
def yolo_loss(pred, target):
    # pred, target: (N, S, S, 5 + C)
    has_obj = target[..., 0] == 1     # (N, S, S): True where a cell is responsible for an object
    p = pred[has_obj]                 # (number of objects, 8)
    t = target[has_obj]
    p_empty = pred[~has_obj]          # (number of empty cells, 8)
    t_empty = target[~has_obj]

    # columns: 0 = conf, 1:3 = x, y, 3:5 = w, h, 5: = class scores
    xy_loss = ((p[:, 1:3] - t[:, 1:3]) ** 2).sum()
    # TODO: same as xy_loss, but for w, h, and take torch.sqrt of both first
    wh_loss = ...
    # TODO: conf of cells with an object should be 1 (hint: t[:, 0] is 1 there)
    obj_loss = ...
    # TODO: conf of empty cells should be 0
    empty_loss = ...
    class_loss = F.cross_entropy(p[:, 5:], t[:, 5:].argmax(dim=1), reduction="sum")

    # TODO: add everything up with the weights 5 and 0.5
    loss = ...
    return loss / len(pred)           # average over the images in the batch

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def yolo_loss(pred, target):
    # pred, target: (N, S, S, 5 + C)
    has_obj = target[..., 0] == 1     # (N, S, S): True where a cell is responsible for an object
    p = pred[has_obj]                 # (number of objects, 8)
    t = target[has_obj]
    p_empty = pred[~has_obj]          # (number of empty cells, 8)   (~ means "not")
    t_empty = target[~has_obj]

    # columns: 0 = conf, 1:3 = x, y, 3:5 = w, h, 5: = class scores
    xy_loss = ((p[:, 1:3] - t[:, 1:3]) ** 2).sum()
    wh_loss = ((torch.sqrt(p[:, 3:5]) - torch.sqrt(t[:, 3:5])) ** 2).sum()
    obj_loss = ((p[:, 0] - t[:, 0]) ** 2).sum()
    empty_loss = ((p_empty[:, 0] - t_empty[:, 0]) ** 2).sum()
    class_loss = F.cross_entropy(p[:, 5:], t[:, 5:].argmax(dim=1), reduction="sum")

    loss = 5 * (xy_loss + wh_loss) + obj_loss + 0.5 * empty_loss + class_loss
    return loss / len(pred)           # average over the images in the batch

In [ ]:
# run this cell to check your code
target = torch.stack([encode_target([[0.5, 0.2, 0.3, 0.1]], [2]), encode_target([[0.1, 0.9, 0.2, 0.2], [0.8, 0.5, 0.4, 0.6]], [0, 1])])
pred = torch.linspace(0.05, 0.95, target.numel()).reshape(target.shape)
print("loss:", round(yolo_loss(pred, target).item(), 4))
print("correct:", abs(yolo_loss(pred, target).item() - 12.5009) < 1e-3)

### Train

The training loop is the same as in Lab 5: **forward → loss → backward → update**. It takes about a minute on a GPU.
Watch the loss go down.

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total = 0
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        pred = model(images)
        loss = yolo_loss(pred, targets)
        optimizer.zero_grad()   # clear old gradients
        loss.backward()         # backpropagate
        optimizer.step()        # update the weights
        total += loss.item() * len(images)
    return total / len(loader.dataset)


torch.manual_seed(0)
model = TinyYOLO().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
losses = []
for epoch in range(15):
    losses.append(train_one_epoch(model, train_loader, optimizer))
    print(f"epoch {epoch + 1:2d} | loss {losses[-1]:.3f}")

plt.plot(range(1, len(losses) + 1), losses)
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.show()

### What does the network see?

This is the famous figure from the YOLO paper, made with **your** network on a test image:
1. the grid,
2. **class map**: every cell colored by the class it predicts, faded if `conf` is low,
3. **all boxes** with `conf` > 0.05 (thicker = more confident),
4. only the boxes with `conf` > 0.5.

In [ ]:
def predict(model, image):
    # image: (3, IMG, IMG) tensor. Returns the (S, S, 5 + C) grid
    model.eval()
    with torch.no_grad():
        grid = model(image.unsqueeze(0).to(device))[0].cpu()
    return grid


i = 8
grid = predict(model, x_test[i])

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax in axes:
    ax.imshow(test_images[i])
    ax.axis("off")
for k in range(1, S):
    axes[0].axhline(k * IMG / S, color="black", linewidth=0.5)
    axes[0].axvline(k * IMG / S, color="black", linewidth=0.5)
axes[0].set_title("1. S x S grid")

class_map = np.zeros((S, S, 4))
for row in range(S):
    for col in range(S):
        class_map[row, col, :3] = plt.cm.colors.to_rgb(COLORS[grid[row, col, 5:].argmax()])
        class_map[row, col, 3] = 0.15 + 0.7 * grid[row, col, 0].item()
axes[1].imshow(class_map, extent=(0, IMG, IMG, 0))
axes[1].set_title("2. class map (" + ", ".join(f"{n}={c[4:]}" for n, c in zip(CLASSES, COLORS)) + ")", fontsize=9)

boxes, labels, scores = decode(grid, conf_threshold=0.05)
for (x, y, w, h), c, s in zip(boxes, labels, scores):
    x1, y1, x2, y2 = [v * IMG for v in xywh_to_xyxy([x, y, w, h])]
    axes[2].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=COLORS[c], linewidth=4 * s))
axes[2].set_title(f"3. all boxes with conf > 0.05 ({len(boxes)})")

boxes, labels, scores = decode(grid, conf_threshold=0.5)
draw_boxes(axes[3], boxes, labels, scores)
axes[3].set_title(f"4. conf > 0.5 ({len(boxes)})")
plt.show()

Look at panel 3: some objects have a second, thinner box. The center of the big circle is close to the line between
two cells, so the neighbor cell also found it (with low confidence). That gives **two boxes for one object**. We fix that in Part 6.
**Try it:** change `i` to look at other test images.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for i, ax in enumerate(axes.flat):
    boxes, labels, scores = decode(predict(model, x_test[i]), conf_threshold=0.5)
    ax.imshow(test_images[i])
    draw_boxes(ax, boxes, labels, scores)
    ax.axis("off")
plt.suptitle("test images (never seen during training)")
plt.tight_layout()
plt.show()

### How good is it?

A predicted box is **correct** if it has the right class **and** IoU ≥ 0.5 with a true box (your `iou` at work!).

**Example** (picture below): an image has 3 true shapes. The model draws 4 boxes: 2 are correct, 1 has the wrong class, 1 is on empty background.
So it found 2 of the 3 shapes and missed 1.

- **Precision** = correct boxes / all predicted boxes = 2 / 4 = **0.5**.
  *When the model draws a box, how often is it right?* Low precision = many wrong boxes (false alarms).
- **Recall** = objects found / all true objects = 2 / 3 = **0.67**.
  *How many of the real objects did the model find?* Low recall = many missed objects.

We want both to be high. A model that draws boxes everywhere finds everything (high recall) but most boxes are wrong
(low precision). A model that draws only one very sure box is usually right (high precision) but misses the rest (low recall).

Below: the example as a picture, then the counts over all 300 test images.

In [ ]:
# the example from the text: 3 true shapes, 4 predicted boxes
true_boxes = [[0.25, 0.3, 0.3, 0.3], [0.72, 0.3, 0.28, 0.28], [0.5, 0.75, 0.32, 0.32]]
true_labels = [0, 1, 2]                     # circle, square, triangle
pred_boxes = [[0.26, 0.31, 0.3, 0.28],      # circle on the circle
              [0.70, 0.32, 0.26, 0.30],     # square on the square
              [0.5, 0.74, 0.3, 0.3],        # "circle" on the triangle: wrong class
              [0.15, 0.82, 0.18, 0.18]]     # "square" on empty background
pred_labels = [0, 1, 0, 1]

# draw the shapes
img = Image.new("RGB", (IMG, IMG), (235, 235, 235))
draw = ImageDraw.Draw(img)
for box, c in zip(true_boxes, true_labels):
    x1, y1, x2, y2 = [v * IMG for v in xywh_to_xyxy(box)]
    if c == 0:
        draw.ellipse([x1, y1, x2, y2], fill=(90, 60, 120))
    elif c == 1:
        draw.rectangle([x1, y1, x2, y2], fill=(90, 60, 120))
    else:
        draw.polygon([((x1 + x2) / 2, y1), (x1, y2), (x2, y2)], fill=(90, 60, 120))


def same_object(box1, c1, box2, c2):
    # correct match = same class and IoU >= 0.5
    return c1 == c2 and iou(xywh_to_xyxy(box1), xywh_to_xyxy(box2)) >= 0.5


fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
for ax in axes:
    ax.imshow(img)
    ax.axis("off")

# left: every predicted box, green = correct, red = wrong
n_correct = 0
for box, c in zip(pred_boxes, pred_labels):
    correct = False
    for true_box, true_c in zip(true_boxes, true_labels):
        if same_object(box, c, true_box, true_c):
            correct = True
    n_correct += correct
    x1, y1, x2, y2 = [v * IMG for v in xywh_to_xyxy(box)]
    color = "green" if correct else "red"
    axes[0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.5))
    axes[0].text(x1, y1 - 2, CLASSES[c] + (" (correct)" if correct else " (wrong)"), color=color, fontsize=9, fontweight="bold",
                 backgroundcolor="white")
axes[0].set_title(f"PRECISION: look at the predicted boxes\n{n_correct} of {len(pred_boxes)} are correct -> "
                  f"precision = {n_correct}/{len(pred_boxes)} = {n_correct / len(pred_boxes):.2f}")

# right: every true object, green = found, red = missed
n_found = 0
for true_box, true_c in zip(true_boxes, true_labels):
    found = False
    for box, c in zip(pred_boxes, pred_labels):
        if same_object(box, c, true_box, true_c):
            found = True
    n_found += found
    x1, y1, x2, y2 = [v * IMG for v in xywh_to_xyxy(true_box)]
    color = "green" if found else "red"
    axes[1].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.5, linestyle="--"))
    axes[1].text(x1, y1 - 2, CLASSES[true_c] + (" (found)" if found else " (missed)"), color=color, fontsize=9, fontweight="bold",
                 backgroundcolor="white")
axes[1].set_title(f"RECALL: look at the true objects\n{n_found} of {len(true_boxes)} were found -> "
                  f"recall = {n_found}/{len(true_boxes)} = {n_found / len(true_boxes):.2f}")
plt.show()

In [ ]:
def is_match(box, c, other_boxes, other_labels):
    # True if box (class c) has IoU >= 0.5 with one of the other boxes of the same class
    for other_box, other_c in zip(other_boxes, other_labels):
        if c == other_c and iou(xywh_to_xyxy(box), xywh_to_xyxy(other_box)) >= 0.51:
            return True
    return False


def evaluate(model, conf_threshold=0.5):
    n_pred, n_correct, n_true, n_found = 0, 0, 0, 0
    for i in range(len(x_test)):
        boxes, labels, scores = decode(predict(model, x_test[i]), conf_threshold)
        for box, c in zip(boxes, labels):                        # precision: is each predicted box correct?
            n_pred += 1
            n_correct += is_match(box, c, test_boxes[i], test_labels[i])
        for box, c in zip(test_boxes[i], test_labels[i]):        # recall: was each true object found?
            n_true += 1
            n_found += is_match(box, c, boxes, labels)
    print(f"precision: {n_correct / max(n_pred, 1):.2f}   recall: {n_found / n_true:.2f}")


evaluate(model)

**Think:** change `conf_threshold` to 0.1 and to 0.9. What happens to precision and recall? Why?

## 6. Non-max suppression (NMS)

You saw it in Part 5: sometimes **several boxes find the same object**. On real photos this happens a lot: a big
object (like the bus) covers many cells, and real YOLO predicts several boxes per cell.
Here is the real YOLO from Part 1 again, with its cleanup step turned off:

In [ ]:
raw = yolo(photo, conf=0.25, iou=1.0, verbose=False)[0]   # iou=1.0 turns NMS off
raw_boxes = raw.boxes.xyxy.tolist()     # corner format, in pixels
raw_scores = raw.boxes.conf.tolist()
raw_labels = raw.boxes.cls.int().tolist()
print("number of boxes:", len(raw_boxes))
for c in set(raw_labels):
    print(f"  {raw.names[c]}: {raw_labels.count(c)} boxes")


def show_photo_boxes(boxes, labels, title):
    plt.figure(figsize=(6, 8))
    plt.imshow(photo)
    for (x1, y1, x2, y2), c in zip(boxes, labels):
        plt.gca().add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                              edgecolor=plt.cm.tab10(c % 10), linewidth=1))
    plt.title(title)
    plt.axis("off")
    plt.show()


show_photo_boxes(raw_boxes, raw_labels, f"before NMS: {len(raw_boxes)} boxes (many lie on top of each other)")

**NMS** keeps the best box and removes the boxes that overlap it too much:

1. Sort the boxes by score, highest first.
2. **Keep** the first (best) box.
3. **Remove** every other box whose IoU with it is above a threshold (e.g. 0.5): they found the same object.
4. Repeat with the boxes that are left, until none are left.

We do this for **each class separately** (a person box should not remove a bus box).

In [ ]:
# a hand-made example: 2 circles, 5 predicted "circle" boxes (center format) with their scores
nms_boxes = [[0.30, 0.40, 0.35, 0.35],   # 0: left circle
             [0.34, 0.44, 0.35, 0.35],   # 1: left circle
             [0.26, 0.44, 0.35, 0.35],   # 2: left circle
             [0.72, 0.68, 0.30, 0.30],   # 3: right circle
             [0.70, 0.72, 0.30, 0.30]]   # 4: right circle
nms_scores = [0.90, 0.75, 0.60, 0.80, 0.50]
label_pos = ["top", "bottom", "left", "top", "bottom"]   # where to write each score, so they don't overlap

img = Image.new("RGB", (IMG, IMG), (235, 235, 235))
draw = ImageDraw.Draw(img)
for x, y, s in [(0.30, 0.40, 0.33), (0.72, 0.68, 0.28)]:
    draw.ellipse([v * IMG for v in xywh_to_xyxy([x, y, s, s])], fill=(90, 60, 120))


def draw_one(ax, k, color, text, style="-"):
    x1, y1, x2, y2 = [v * IMG for v in xywh_to_xyxy(nms_boxes[k])]
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.5, linestyle=style))
    if label_pos[k] == "top":
        ax.text(x1, y1 - 2, text, color=color, fontsize=8, fontweight="bold", backgroundcolor="white")
    elif label_pos[k] == "bottom":
        ax.text(x2, y2 + 8, text, color=color, fontsize=8, fontweight="bold", backgroundcolor="white", ha="right")
    else:
        ax.text(x1 - 2, (y1 + y2) / 2, text, color=color, fontsize=8, fontweight="bold", backgroundcolor="white", ha="right")


def iou_with(k, best):
    return iou(xywh_to_xyxy(nms_boxes[k]), xywh_to_xyxy(nms_boxes[best]))


fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for ax in axes:
    ax.imshow(img)
    ax.set_xlim(-45, IMG + 5)
    ax.set_ylim(IMG + 12, -8)
    ax.axis("off")

# start: all boxes, sorted by score
start_colors = ["tab:blue", "tab:orange", "tab:purple", "tab:brown", "tab:pink"]
for k in range(5):
    draw_one(axes[0], k, start_colors[k], f"{nms_scores[k]:.2f}")
axes[0].set_title("Start: 5 boxes\nsorted by score: 0.90, 0.80, 0.75, 0.60, 0.50")

# round 1: keep 0.90 (box 0). Compare every other box with it
draw_one(axes[1], 0, "green", "0.90 keep")
for k in [1, 2, 3, 4]:
    if iou_with(k, 0) > 0.5:
        draw_one(axes[1], k, "red", f"{nms_scores[k]:.2f} remove (IoU {iou_with(k, 0):.2f})", "--")
    else:
        draw_one(axes[1], k, "gray", f"{nms_scores[k]:.2f} stays (IoU {iou_with(k, 0):.2f})")
axes[1].set_title("Round 1: keep the best (0.90)\nremove boxes with IoU > 0.5 with it")

# round 2: boxes left: 0.80 (box 3) and 0.50 (box 4). Keep 0.80, compare 0.50 with it
draw_one(axes[2], 0, "green", "0.90 kept")
draw_one(axes[2], 3, "green", "0.80 keep")
draw_one(axes[2], 4, "red", f"0.50 remove (IoU {iou_with(4, 3):.2f})", "--")
axes[2].set_title("Round 2: best box left is 0.80, keep it\nremove boxes with IoU > 0.5 with it")

# done
draw_one(axes[3], 0, "green", "0.90")
draw_one(axes[3], 3, "green", "0.80")
axes[3].set_title("Done: no boxes left to check\n2 boxes, one per object")
plt.tight_layout()
plt.show()

**Andrew Ng: Non-max Suppression (optional)** (click to watch)

<a href="https://www.youtube.com/watch?v=VAo84c1hQX8" target="_blank"><img src="https://img.youtube.com/vi/VAo84c1hQX8/hqdefault.jpg" width="480"></a>

**TODO 7:** write `nms`. It returns the **indices** of the boxes to keep.

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    # boxes: list of [x1, y1, x2, y2], scores: list of numbers
    order = np.argsort(scores)[::-1].tolist()   # indices of the boxes, best score first
    keep = []
    while len(order) > 0:
        # TODO: the best box left is the first one in order. Keep it.
        best = ...
        ...
        # TODO: keep in "order" only the other boxes whose IoU with the best box is <= iou_threshold
        remaining = []
        for i in order[1:]:
            ...
        order = remaining
    return keep

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def nms(boxes, scores, iou_threshold=0.5):
    # boxes: list of [x1, y1, x2, y2], scores: list of numbers
    order = np.argsort(scores)[::-1].tolist()   # indices of the boxes, best score first
    keep = []
    while len(order) > 0:
        best = order[0]
        keep.append(best)
        remaining = []
        for i in order[1:]:
            if iou(boxes[best], boxes[i]) <= iou_threshold:
                remaining.append(i)
        order = remaining
    return keep

In [ ]:
# run this cell to check your code (compare with torchvision)
rng = np.random.default_rng(0)
xy = rng.random((50, 2)) * 0.5
test_b = np.concatenate([xy, xy + 0.2 + rng.random((50, 2)) * 0.3], axis=1).tolist()
test_s = rng.random(50).tolist()
mine = nms(test_b, test_s, 0.5)
real = torchvision.ops.nms(torch.tensor(test_b), torch.tensor(test_s), 0.5).tolist()
print("kept", len(mine), "of 50 boxes")
print("correct:", mine == real)

In [ ]:
def nms_per_class(boxes, scores, labels, iou_threshold=0.5):
    keep = []
    for c in set(labels):
        idx = [i for i in range(len(boxes)) if labels[i] == c]        # boxes of this class
        kept = nms([boxes[i] for i in idx], [scores[i] for i in idx], iou_threshold)
        keep += [idx[k] for k in kept]
    return keep


keep = nms_per_class(raw_boxes, raw_scores, raw_labels)
show_photo_boxes([raw_boxes[i] for i in keep], [raw_labels[i] for i in keep],
                 f"after your NMS: {len(keep)} boxes")
print("your NMS:", [(raw.names[raw_labels[i]], round(raw_scores[i], 2)) for i in keep])

That is what real YOLO does too (compare with the picture in Part 1).

**Try it:** change `iou_threshold` in `nms_per_class` to 0.1 and to 0.9. What goes wrong in each case?

Now use your NMS on **your** tiny YOLO, on the same test image as in Part 5. We keep all boxes with conf > 0.05, so the weak extra boxes are there too:

In [ ]:
i = 8
boxes, labels, scores = decode(predict(model, x_test[i]), conf_threshold=0.05)
keep = nms_per_class([xywh_to_xyxy(b) for b in boxes], scores, labels)   # nms needs corner format

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax in axes:
    ax.imshow(test_images[i])
    ax.axis("off")
draw_boxes(axes[0], boxes, labels, scores)
draw_boxes(axes[1], [boxes[k] for k in keep], [labels[k] for k in keep], [scores[k] for k in keep])
axes[0].set_title(f"before NMS: {len(boxes)} boxes")
axes[1].set_title(f"after NMS: {len(keep)} boxes")
plt.show()

## 7. Try it yourself

Let's visit Satriale's Pork Store from *The Sopranos*. Who and what does YOLO find? Look at the people in the back and the
tie: how confident is it there?

Upload your own photo (in Colab: the folder icon on the left), change the file name below and run.
`yolo` knows 80 classes (`print(yolo.names)` to see them). What does it miss? What does it get wrong?

*Image: still from "The Sopranos" (HBO), [CC BY 3.0](https://creativecommons.org/licenses/by/3.0), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Satriale%27s_(The_Sopranos_Family_Tree).jpg).*

In [ ]:
import os
import urllib.request

IMG_URL = "https://raw.githubusercontent.com/arash-mham/visual-computing-I/main/lab-material/lab6/sopranos.jpg"
if not os.path.exists("sopranos.jpg"):
    urllib.request.urlretrieve(IMG_URL, "sopranos.jpg")

my_photo = Image.open("sopranos.jpg")   # TODO (optional): change to your own file, e.g. Image.open("my_photo.jpg")
result = yolo(my_photo, verbose=False)[0]
plt.figure(figsize=(12, 7))
plt.imshow(result.plot()[:, :, ::-1])
plt.axis("off")
plt.show()

## Summary

- **Detection** = for every object: class + box + confidence.
- **YOLO** cuts the image into an $S \times S$ grid. The cell with the object's **center** is responsible for it and
  predicts `conf, x, y, w, h` and the class. One CNN pass gives all boxes: **You Only Look Once**.
- **IoU** = overlap / union. It measures how good a box is (≥ 0.5 usually counts as correct).
- **Loss:** squared errors on box and confidence (with weights 5 and 0.5), plus a class loss.
- **NMS** removes duplicate boxes: keep the best, remove the ones that overlap it, repeat.

**Limits of YOLOv1** (and how later versions fixed them): one object per cell, so groups of small objects
(birds, people in a crowd) are hard. Later YOLOs use several grids of different sizes and several "anchor" box shapes per cell.

**Watch at home:** the creator of YOLO explains it in a TED talk.

**Joseph Redmon (TED): How computers learn to recognize objects instantly** (click to watch)

<a href="https://www.youtube.com/watch?v=Cgxsv1riJhI" target="_blank"><img src="https://img.youtube.com/vi/Cgxsv1riJhI/hqdefault.jpg" width="480"></a>